# VLM + Quantum + CFD Coupling Demo for F1 Aero

This notebook demonstrates a practical prototype workflow:
1. Generate NACA-inspired aerodynamic node data for an F1 wing surface.
2. Build VLM-style node metrics (local lift/drag at control points).
3. Encode node-selection into a QUBO optimization problem.
4. Solve classically (and optionally with Qiskit if available).
5. Apply a CFD-like correction loop to estimate final coupled performance.

## Data Notes

- NACA-inspired geometry logic is used for synthetic generation.
- NASA/NACA benchmark datasets can replace this synthetic stage later with the same schema (`x,y,z,lift,drag,cp,gamma`).
- Goal is to validate end-to-end integration behavior before high-fidelity CFD runs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries ready')

In [ ]:
def naca4_thickness(x, t=0.12):
    # Classic NACA 4-digit symmetric thickness distribution
    return 5 * t * (
        0.2969 * np.sqrt(np.maximum(x, 1e-6))
        - 0.1260 * x
        - 0.3516 * x**2
        + 0.2843 * x**3
        - 0.1015 * x**4
    )

n_span = 16
n_chord = 10
span = 1.2
chord = 0.28
velocity = 72.0
alpha_deg = 4.5
alpha = np.radians(alpha_deg)
rho = 1.225
q_inf = 0.5 * rho * velocity**2

rows = []
node_id = 0
for i in range(n_span):
    span_eta = i / (n_span - 1)
    y = (span_eta - 0.5) * span
    span_loading = np.sqrt(max(0.0, 1 - ((2 * span_eta - 1)**2)))

    for j in range(n_chord):
        x_eta = (j + 0.5) / n_chord
        x = x_eta * chord
        z = naca4_thickness(x_eta, t=0.12) * chord

        gamma = 0.42 * span_loading * (1.2 - 0.7 * x_eta)
        cp = -1.6 * gamma + 0.04 * np.random.randn()

        local_lift = q_inf * (0.010 + 0.020 * gamma + 0.002 * alpha_deg)
        local_drag = q_inf * (0.0015 + 0.0018 * abs(cp) + 0.0007 * (x_eta**2))

        rows.append({
            'node_id': node_id,
            'span_index': i,
            'chord_index': j,
            'x': x,
            'y': y,
            'z': z,
            'gamma': gamma,
            'cp': cp,
            'lift': local_lift,
            'drag': local_drag
        })
        node_id += 1

nodes = pd.DataFrame(rows)
nodes['score'] = nodes['lift'] - nodes['drag']
nodes.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sc0 = ax[0].scatter(nodes['y'], nodes['x'], c=nodes['lift'], cmap='viridis', s=30)
ax[0].set_title('VLM Nodes Colored by Local Lift')
ax[0].set_xlabel('Spanwise y [m]')
ax[0].set_ylabel('Chordwise x [m]')
fig.colorbar(sc0, ax=ax[0], label='Lift [N]')

sc1 = ax[1].scatter(nodes['y'], nodes['x'], c=nodes['drag'], cmap='magma', s=30)
ax[1].set_title('VLM Nodes Colored by Local Drag')
ax[1].set_xlabel('Spanwise y [m]')
ax[1].set_ylabel('Chordwise x [m]')
fig.colorbar(sc1, ax=ax[1], label='Drag [N]')

plt.tight_layout()

In [ ]:
# Build a QUBO for node selection: minimize drag - lift + smoothness penalties
subset = nodes.nlargest(14, 'score').reset_index(drop=True).copy()
N = len(subset)
drag_w = 1.0
lift_w = 1.0
coupling_w = 0.02

lift_norm = subset['lift'] / subset['lift'].max()
drag_norm = subset['drag'] / subset['drag'].max()

Q = np.zeros((N, N))
for i in range(N):
    Q[i, i] = drag_w * drag_norm.iloc[i] - lift_w * lift_norm.iloc[i]

for i in range(N):
    for j in range(i + 1, N):
        dy = subset.loc[i, 'y'] - subset.loc[j, 'y']
        dx = subset.loc[i, 'x'] - subset.loc[j, 'x']
        dist = np.sqrt(dx**2 + dy**2)
        c = coupling_w / (1 + dist)
        Q[i, j] = c
        Q[j, i] = c

def qubo_cost(x, Q):
    x = np.asarray(x)
    return float(x @ Q @ x)

# Exhaustive classical solve for this small subset
best_cost = 1e9
best_x = None
for state in range(2**N):
    x = np.array(list(np.binary_repr(state, width=N)), dtype=int)
    cost = qubo_cost(x, Q)
    if cost < best_cost:
        best_cost = cost
        best_x = x

subset['selected'] = best_x
selected = subset[subset['selected'] == 1]

print(f'Variables: {N}')
print(f'Best cost: {best_cost:.6f}')
print(f'Selected nodes: {len(selected)}')
selected[['node_id', 'lift', 'drag', 'score']].head()

In [ ]:
# Optional: Qiskit path (if installed in the environment)
try:
    from qiskit_optimization import QuadraticProgram
    from qiskit_optimization.algorithms import MinimumEigenOptimizer
    from qiskit_algorithms import QAOA
    from qiskit_algorithms.optimizers import COBYLA
    from qiskit.primitives import Sampler

    qp = QuadraticProgram('vlm_node_selection')
    for i in range(N):
        qp.binary_var(name=f'x{i}')

    linear = {f'x{i}': float(Q[i, i]) for i in range(N)}
    quadratic = {}
    for i in range(N):
        for j in range(i + 1, N):
            if abs(Q[i, j]) > 1e-12:
                quadratic[(f'x{i}', f'x{j}')] = float(Q[i, j])

    qp.minimize(linear=linear, quadratic=quadratic)

    qaoa = QAOA(sampler=Sampler(), optimizer=COBYLA(maxiter=80), reps=2)
    qoptimizer = MinimumEigenOptimizer(qaoa)
    q_result = qoptimizer.solve(qp)

    quantum_x = np.array([int(q_result.variables_dict[f'x{i}']) for i in range(N)])
    print('Quantum solution objective:', q_result.fval)
    print('Quantum selected nodes:', quantum_x.sum())
except Exception as exc:
    print('Qiskit optional path not available:', exc)

In [ ]:
# Coupled CFD proxy: blend VLM estimate with correction based on selected-node topology
baseline_cl = nodes['lift'].sum() / (q_inf * span * chord)
baseline_cd = nodes['drag'].sum() / (q_inf * span * chord)

selection_ratio = len(selected) / max(len(subset), 1)
opt_cl = baseline_cl * (1 + 0.06 * selection_ratio)
opt_cd = baseline_cd * (1 - 0.08 * selection_ratio)

# CFD correction proxy: adds turbulence/yaw sensitivity
yaw_deg = 0.5
cfd_cl = 0.7 * opt_cl + 0.3 * baseline_cl
cfd_cd = (0.7 * opt_cd + 0.3 * baseline_cd) * (1 + 0.01 * abs(yaw_deg))

labels = ['Baseline VLM', 'Optimized (QUBO)', 'CFD Coupled Proxy']
cl_values = [baseline_cl, opt_cl, cfd_cl]
cd_values = [baseline_cd, opt_cd, cfd_cd]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(labels, cl_values, color=['#64748b', '#2563eb', '#16a34a'])
ax[0].set_title('CL Comparison')
ax[0].tick_params(axis='x', rotation=20)

ax[1].bar(labels, cd_values, color=['#64748b', '#2563eb', '#16a34a'])
ax[1].set_title('CD Comparison')
ax[1].tick_params(axis='x', rotation=20)

plt.tight_layout()

print('Baseline L/D:', baseline_cl / baseline_cd)
print('Optimized L/D:', opt_cl / opt_cd)
print('CFD-coupled L/D:', cfd_cl / cfd_cd)